# DigiSteel-YOLO: Phase 1 Ablation Study

## Goal: Beat P04, P05, P08, P09 (mAP@0.5 > 80%)

**Experiments (sequential, ~10-12 hours on T4):**
1. C3Ghost Architecture (100 epochs)
2. Image Size 800 (100 epochs)
3. Image Size 1280 (100 epochs)
4. Enhanced Augmentation (100 epochs)
5. Cosine Learning Rate (100 epochs)
6. Final Optimized Model (200 epochs)

**Features:** Auto-GitHub push, OOM recovery, resume capability

**Instructions:**
1. Runtime → Change runtime type → T4 GPU → Save
2. Run → Run all cells
3. Go to sleep. Wake up to results in GitHub.

**Reference Papers to Beat:**
| Paper | mAP@0.5 | Status |
|---|---|---|
| P04 Lightweight-YOLOv8 | 78.6% | Target |
| P05 SCCI-YOLO | 78.6% | Target |
| P08 MSFE-YOLO | 79.8% | Target |
| P09 EFEN-YOLOv8 | 80.4% | Target |
| **Our Baseline** | **77.9%** | Starting point |

In [ ]:
#@title Cell 1: Setup Environment

import os, json, time, subprocess
from pathlib import Path

# ---- GPU CHECK (fail fast) ----
import torch
assert torch.cuda.is_available(), 'NO GPU! Runtime > Change runtime type > T4 GPU > Save > Re-run'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEM:.1f} GB)')
print(f'CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}')

# ---- Install dependencies ----
!pip install -q ultralytics albumentations

# ---- Configure Kaggle ----
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
creds = {'username': 'hazemelerefy', 'key': 'KGAT_0e5696318d7e5a3caf038db9497466e5'}
(kaggle_dir / 'kaggle.json').write_text(json.dumps(creds))
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']

# ---- Clone repo & pull latest ----
os.chdir('/content')
if not Path('DigiSteel-YOLO').exists():
    !git clone https://github.com/hazemelerefey/DigiSteel-YOLO.git
os.chdir('DigiSteel-YOLO')
!git pull

# ---- Download dataset ----
if not Path('datasets/NEU-DET/yolo/images/train').exists():
    !kaggle datasets download -d sovitrath/neu-steel-surface-defect-detect-trainvalid-split -p datasets/NEU-DET --unzip -q
    !python tools/prepare_datasets.py

print('\nEnvironment ready!')

In [ ]:
#@title Cell 2: Run Ablation Study (5 experiments, ~8 hours)

import json, time, subprocess
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import torch

# ---- Config ----
DATASET = 'NEU-DET'
SEED = 42
CONFIG_PATH = f'configs/{DATASET.lower().replace("-", "_")}.yaml'
RESULTS_DIR = Path('evals')
PROGRESS_FILE = RESULTS_DIR / 'ablation_progress.json'
RESULTS_DIR.mkdir(exist_ok=True)

def load_progress():
    if PROGRESS_FILE.exists():
        return json.load(open(PROGRESS_FILE))
    return {'completed': {}, 'best_map50': 0.0, 'best_experiment': None}

def save_progress(progress):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, indent=2)

def push_to_github(msg):
    try:
        subprocess.run(['git', 'add', '-A'], capture_output=True, timeout=30)
        subprocess.run(['git', 'commit', '-m', f'[auto] {msg}'], capture_output=True, timeout=30)
        r = subprocess.run(['git', 'push'], capture_output=True, timeout=60, text=True)
        print(f'  Pushed: {msg}' if r.returncode == 0 else f'  Push failed: {r.stderr[:100]}')
    except Exception as e:
        print(f'  Push error: {e}')

def train_and_evaluate(name, model_config, imgsz=640, batch=16,
                       cos_lr=False, mosaic=1.0, mixup=0.0, copy_paste=0.0,
                       epochs=100, patience=30, lr0=0.01):
    run_name = f'ablation_{name.lower().replace(" ", "_")}_{DATASET.lower()}_seed{SEED}'
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {name}')
    print(f'  Model: {model_config} | imgsz: {imgsz} | batch: {batch}')
    print(f'  Epochs: {epochs} | cos_lr: {cos_lr} | mixup: {mixup} | lr0: {lr0}')
    print(f'{"="*60}')

    model = YOLO(model_config)
    start = time.time()
    try:
        model.train(
            data=CONFIG_PATH, epochs=epochs, imgsz=imgsz, batch=batch,
            seed=SEED, project='runs', name=run_name, exist_ok=True,
            verbose=True, patience=patience, cos_lr=cos_lr,
            mosaic=mosaic, mixup=mixup, copy_paste=copy_paste,
            lr0=lr0, lrf=0.01, momentum=0.937, weight_decay=0.0005,
            warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
            close_mosaic=10, amp=True,
        )
        elapsed = time.time() - start
        csv = Path(f'runs/{run_name}/results.csv')
        if csv.exists():
            df = pd.read_csv(csv)
            mAP50 = float(df['metrics/mAP50(B)'].max())
            mAP50_95 = float(df['metrics/mAP50-95(B)'].max())
            best_ep = int(df['metrics/mAP50(B)'].idxmax())
            print(f'  mAP@0.5: {mAP50:.2%} | mAP@0.5:0.95: {mAP50_95:.2%} | Best epoch: {best_ep}')
            return {'experiment': name, 'mAP50': mAP50, 'mAP50_95': mAP50_95,
                    'best_epoch': best_ep, 'time_hours': elapsed/3600,
                    'imgsz': imgsz, 'batch': batch, 'epochs': epochs, 'model': model_config}
        print(f'  Results CSV not found!')
        return None
    except torch.cuda.OutOfMemoryError:
        print(f'  OOM! Reducing batch {batch} -> {max(4, batch//2)} and retrying...')
        torch.cuda.empty_cache()
        return train_and_evaluate(name, model_config, imgsz, max(4, batch//2),
                                   cos_lr, mosaic, mixup, copy_paste, epochs, patience, lr0)
    except Exception as e:
        print(f'  Error: {e}')
        return None

# ---- Create C3Ghost config ----
Path('configs').mkdir(exist_ok=True)
Path('configs/yolov11n_c3ghost.yaml').write_text('''nc: 6
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 1, C3Ghost, [128]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 1, C3Ghost, [256]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, C3Ghost, [512]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 1, C3Ghost, [1024]]
  - [-1, 1, SPPF, [1024, 5]]
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [1024, False]]
  - [[15, 18, 21], 1, Detect, [nc]]
''')

# ---- Experiments ----
EXPERIMENTS = [
    {'name': 'C3Ghost_Arch', 'model': 'configs/yolov11n_c3ghost.yaml', 'imgsz': 640, 'batch': 16, 'lr0': 0.01},
    {'name': 'ImgSize_800',  'model': 'yolo11n.pt', 'imgsz': 800,  'batch': 12, 'lr0': 0.008},
    {'name': 'ImgSize_1280', 'model': 'yolo11n.pt', 'imgsz': 1280, 'batch': 8,  'lr0': 0.005},
    {'name': 'Enhanced_Aug', 'model': 'yolo11n.pt', 'imgsz': 640,  'batch': 16, 'lr0': 0.01,  'mixup': 0.15, 'copy_paste': 0.15},
    {'name': 'Cosine_LR',    'model': 'yolo11n.pt', 'imgsz': 640,  'batch': 16, 'lr0': 0.01,  'cos_lr': True},
]

# ---- Run ----
progress = load_progress()
print(f'GPU: {GPU_NAME} | Dataset: {DATASET} | Seed: {SEED}')
print(f'Already completed: {list(progress["completed"].keys())}')
print(f'Current best: {progress["best_experiment"]} ({progress["best_map50"]:.2%})')

for exp in EXPERIMENTS:
    if exp['name'] in progress['completed']:
        print(f'\n  Skip {exp["name"]} (done: {progress["completed"][exp["name"]]["mAP50"]:.2%})')
        continue
    result = train_and_evaluate(
        name=exp['name'], model_config=exp['model'],
        imgsz=exp['imgsz'], batch=exp['batch'],
        cos_lr=exp.get('cos_lr', False),
        mosaic=exp.get('mosaic', 1.0),
        mixup=exp.get('mixup', 0.0),
        copy_paste=exp.get('copy_paste', 0.0),
        epochs=100, patience=30, lr0=exp.get('lr0', 0.01),
    )
    if result:
        progress['completed'][exp['name']] = result
        if result['mAP50'] > progress['best_map50']:
            progress['best_map50'] = result['mAP50']
            progress['best_experiment'] = exp['name']
            print(f'  NEW BEST: {exp["name"]} @ {result["mAP50"]:.2%}')
        save_progress(progress)
        push_to_github(f'{exp["name"]}: mAP@0.5={result["mAP50"]:.2%}')

# ---- Summary ----
print(f'\n{"="*60}')
print('ABLATION STUDY COMPLETE')
print(f'{"="*60}')
for name, data in progress['completed'].items():
    print(f'  {name}: mAP@0.5={data["mAP50"]:.2%} | mAP@0.5:0.95={data["mAP50_95"]:.2%}')
print(f'\nBest: {progress["best_experiment"]} @ {progress["best_map50"]:.2%}')

In [ ]:
#@title Cell 3: Train Final Optimized Model (200 epochs, ~2.5 hours)

import json, time
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import torch

progress = json.load(open('evals/ablation_progress.json'))
best_name = progress['best_experiment']
best_data = progress['completed'][best_name]

print(f'Best config: {best_name} @ {best_data["mAP50"]:.2%}')
print(f'Training final model for 200 epochs...')

model_config = best_data['model']
imgsz = best_data['imgsz']
batch = max(4, best_data['batch'] - 2)  # Slightly smaller for stability

run_name = f'final_digisteel_{DATASET.lower()}_seed{SEED}'
model = YOLO(model_config)

start = time.time()
try:
    model.train(
        data=CONFIG_PATH, epochs=200, imgsz=imgsz, batch=batch,
        seed=SEED, project='runs', name=run_name, exist_ok=True,
        verbose=True, patience=50, cos_lr=True,
        mosaic=1.0, mixup=0.15, copy_paste=0.1,
        lr0=0.005, lrf=0.01, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
        close_mosaic=10, amp=True,
    )
    elapsed = time.time() - start

    csv = Path(f'runs/{run_name}/results.csv')
    if csv.exists():
        df = pd.read_csv(csv)
        mAP50 = float(df['metrics/mAP50(B)'].max())
        mAP50_95 = float(df['metrics/mAP50-95(B)'].max())
        best_ep = int(df['metrics/mAP50(B)'].idxmax())

        print(f'\n{"="*60}')
        print('FINAL MODEL RESULTS')
        print(f'{"="*60}')
        print(f'  mAP@0.5: {mAP50:.2%}')
        print(f'  mAP@0.5:0.95: {mAP50_95:.2%}')
        print(f'  Best epoch: {best_ep}')
        print(f'  Training time: {elapsed/3600:.1f} hours')
        print(f'  Weights: runs/{run_name}/weights/best.pt')

        final = {
            'model': 'DigiSteel-YOLO Final',
            'based_on': best_name,
            'mAP50': mAP50, 'mAP50_95': mAP50_95,
            'best_epoch': best_ep, 'time_hours': elapsed/3600,
            'imgsz': imgsz, 'batch': batch, 'epochs': 200,
        }
        with open('evals/final_model_results.json', 'w') as f:
            json.dump(final, f, indent=2)

        # Update progress
        progress['completed']['FINAL'] = final
        if mAP50 > progress['best_map50']:
            progress['best_map50'] = mAP50
            progress['best_experiment'] = 'FINAL'
        with open('evals/ablation_progress.json', 'w') as f:
            json.dump(progress, f, indent=2)

        # Push to GitHub
        subprocess.run(['git', 'add', '-A'], capture_output=True, timeout=30)
        subprocess.run(['git', 'commit', '-m', f'[auto] FINAL MODEL: mAP@0.5={mAP50:.2%}'], capture_output=True, timeout=30)
        subprocess.run(['git', 'push'], capture_output=True, timeout=60)
        print('  Pushed to GitHub!')

except torch.cuda.OutOfMemoryError:
    print(f'OOM! Try reducing batch size. Current: {batch}')
    torch.cuda.empty_cache()
except Exception as e:
    print(f'Error: {e}')

In [ ]:
#@title Cell 4: Generate Report + ONNX Export

import json
from pathlib import Path
from ultralytics import YOLO
import subprocess

# Load results
progress = json.load(open('evals/ablation_progress.json'))
best_name = progress['best_experiment']
best_map = progress['best_map50'] * 100

# Reference papers
papers = [
    ('P10 KDM-YOLO', 95.4), ('P11 YOLOv11-EMD', 94.9), ('P02 LAM-YOLOv10n', 94.39),
    ('P07 ASFRW-YOLO', 83.2), ('P03 YOLO-LSDI', 83.0), ('P09 EFEN-YOLOv8', 80.4),
    ('P08 MSFE-YOLO', 79.8), ('P04 Lightweight-YOLOv8', 78.6), ('P05 SCCI-YOLO', 78.6),
]
beaten = sum(1 for _, mAP in papers if best_map > mAP)

# Generate report
report = f'''# DigiSteel-YOLO Phase 1 - Final Report

'
report += f'## Best Model: {best_name}\n'
report += f'## mAP@0.5: {best_map:.2f}%\n'
report += f'## Papers Beaten: {beaten}/9\n\n'
report += '## All Results\n\n'
report += '| Experiment | mAP@0.5 | mAP@0.5:0.95 | Time |\n'
report += '|---|---|---|---|\n'
report += '| Baseline (YOLOv11n) | 77.90% | 45.00% | 1.3h |\n'
for name, data in sorted(progress['completed'].items()):
    report += f'| {name} | {data["mAP50"]*100:.2f}% | {data["mAP50_95"]*100:.2f}% | {data["time_hours"]:.1f}h |\n'
report += '\n## Paper Comparison\n\n'
report += '| Paper | mAP@0.5 | Our Best | Status |\n'
report += '|---|---|---|---|\n'
for name, mAP in papers:
    diff = best_map - mAP
    status = 'BEATEN' if diff > 0 else 'NOT YET'
    report += f'| {name} | {mAP}% | {best_map:.2f}% | {status} |\n'
report += f'| **Our Baseline** | **77.90%** | — | — |\n'
report += f'| **Our Best ({best_name})** | **{best_map:.2f}%** | — | **BEST** |\n'

Path('evals/ablation_final_report.md').write_text(report)
print(report)

# ONNX export
run_name = f'final_digisteel_{DATASET.lower()}_seed{SEED}'
weights = f'runs/{run_name}/weights/best.pt'
if Path(weights).exists():
    print(f'\nExporting {weights} to ONNX...')
    model = YOLO(weights)
    model.export(format='onnx', imgsz=640, simplify=True)
    print('ONNX exported!')
else:
    print(f'\nWeights not found at {weights}, skipping ONNX export')

# Final push
subprocess.run(['git', 'add', '-A'], capture_output=True, timeout=30)
subprocess.run(['git', 'commit', '-m', '[auto] Phase 1 complete - report + ONNX'], capture_output=True, timeout=30)
subprocess.run(['git', 'push'], capture_output=True, timeout=60)
print('\nAll pushed to GitHub!')